# 05 — Técnicas Utilizadas e Metodologia do Projeto
**Projeto Aplicado III — Mackenzie**
**Tema:** Sistema de Recomendação de Produtos Fitness — Amazon Sports & Outdoors


## 1. Pipeline Completo do Projeto

```
[Sports_and_Outdoors.jsonl]
          │
          ▼
  [Amostragem (150K–200K)]
          │
          ▼
  [Filtragem de qualidade]
   ≥10 reviews/produto · ≥5 reviews/usuário · texto ≥5 tokens
          │
          ▼
  [Limpeza textual]
   Lower case → HTML/URL → stopwords → stemming
          │
          ▼
  [Perfil de produto]
   Concatenação dos textos limpos por ASIN
          │
          ▼
  [TF-IDF (15K features, bigramas, sublinear TF)]
          │
          ▼
  [Divisão temporal 80/20]
          │
          ▼
  [Perfil de usuário = média ponderada por rating]
  [Cosine Similarity vs catálogo]
  [Filtra produtos já consumidos]
          │
          ▼
  [Top-K recomendações]
          │
          ▼
  [Avaliação: Precision@K · Recall@K · HitRate@K]
```


## 2. Técnicas Utilizadas

### 2.1 TF-IDF (Term Frequency — Inverse Document Frequency)

TF-IDF é uma técnica de vetorização textual que pondera cada termo pela sua frequência no documento (TF) e pela raridade no corpus (IDF). Termos frequentes em poucos documentos recebem pesos maiores, capturando melhor as características individuais de cada produto.

**Configuração utilizada:**
- `max_features=15.000` — limita o vocabulário aos termos mais relevantes
- `ngram_range=(1,2)` — inclui unigramas e bigramas (ex: *"running shoes"*)
- `sublinear_tf=True` — aplica log na frequência para suavizar termos muito repetidos
- `min_df=3` — ignora termos que aparecem em menos de 3 documentos
- `max_df=0.85` — ignora termos presentes em mais de 85% dos documentos

### 2.2 Similaridade do Cosseno

Mede o ângulo entre dois vetores no espaço TF-IDF. Valores próximos de 1 indicam alta similaridade textual entre os perfis dos produtos.

### 2.3 Stemming (Porter Stemmer)

Reduz palavras às suas raízes morfológicas (ex: *running → run*, *shoes → shoe*). Melhora o agrupamento semântico e reduz a dimensionalidade do vocabulário.

### 2.4 Perfil de Usuário Ponderado

O vetor de perfil do usuário é construído como a média ponderada dos vetores TF-IDF dos produtos que ele avaliou positivamente (rating ≥ 4), usando o próprio rating como peso. Isso faz com que produtos com notas mais altas contribuam mais para o perfil.

### 2.5 Divisão Temporal (Train/Test Split)

Em vez de divisão aleatória, usamos divisão temporal: os 20% registros mais recentes de cada usuário vão para teste. Isso simula o cenário real onde o modelo é avaliado em interações futuras.


## 3. Métricas de Avaliação

| Métrica | O que mede |
|---|---|
| **Precision@K** | Dos K itens recomendados, qual fração é relevante |
| **Recall@K** | Dos itens relevantes no teste, qual fração foi recuperada |
| **HitRate@K** | % de usuários que receberam pelo menos 1 recomendação correta |

Um item é considerado **relevante** se o usuário o avaliou com **rating ≥ 4** no conjunto de teste.


## 4. Metodologia do Projeto

O projeto seguiu o fluxo padrão de um pipeline de ciência de dados:

**Etapa 1 — Entendimento dos dados**
Análise exploratória do dataset Amazon Sports & Outdoors, identificando distribuição de ratings, volume de avaliações, esparsidade da matriz usuário-produto e características dos textos.

**Etapa 2 — Modelo Baseline**
Implementação inicial simples do sistema content-based com TF-IDF sem bigramas, sem stemming e com perfil de usuário por média simples. Os resultados serviram de linha de base para comparação.

**Etapa 3 — Ajuste e Melhoria**
Aplicação de pré-processamento mais robusto (stemming, limpeza de HTML/URL), TF-IDF com bigramas e perfil de usuário ponderado pelo rating. Comparação direta com o baseline usando as mesmas métricas e protocolo de avaliação.

**Justificativa da abordagem Content-Based:**
A esparsidade da matriz usuário-produto (> 99%) inviabiliza filtros colaborativos puros, que dependem de co-ocorrências entre usuários. A abordagem baseada em conteúdo contorna esse problema ao usar apenas os textos das avaliações, sem precisar de múltiplos usuários por produto.


## 5. Referências

- MCAULEY, J. et al. *Inferring networks of substitutable and complementary products*. ACM SIGKDD, 2015.
- RICCI, F.; ROKACH, L.; SHAPIRA, B. *Recommender Systems Handbook*. Springer, 2011.
- AGGARWAL, C. C. *Recommender Systems: The Textbook*. Springer, 2016.
- Scikit-learn: *TfidfVectorizer documentation*. https://scikit-learn.org
- NLTK: *Natural Language Toolkit*. https://www.nltk.org
